# 05 · Modelos supervisados y SHAP

**Objetivo.** Comparar modelos para (a) predecir `log1p(PLI)` y (b) clasificar
la excedencia de PEL, con validación **agrupada por punto** y **hold-out
temporal**, e interpretar con SHAP.

Regla anti-fuga: los 8 metales que componen el PLI están prohibidos como
entradas (`tests/test_no_leakage.py`).

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "reports" / "tables"
FIGURES = ROOT / "reports" / "figures"

## 1. Regresión del PLI

In [ ]:
reg = pd.read_csv(TABLES / "metricas_cv_regresion_pli.csv", encoding="utf-8-sig")
reg.round(3)

In [ ]:
holdout = pd.read_csv(TABLES / "holdout_regresion_pli.csv", encoding="utf-8-sig")
holdout.round(3)

**Lectura.** El R² en CV agrupada es la referencia honesta: al separar por
punto, el modelo no puede memorizar el punto de muestreo. La caída en el
hold-out temporal cuantifica la deriva entre campañas.

## 2. Clasificación de excedencia de PEL

In [ ]:
clf = pd.read_csv(TABLES / "metricas_cv_clasificacion_riesgo.csv", encoding="utf-8-sig")
clf.round(3)

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(FIGURES / "12_comparacion_modelos_regresion.png")))

## 3. SHAP: qué variables empujan la predicción

In [ ]:
imp = pd.read_csv(TABLES / "shap_importancia_regresion_pli.csv", encoding="utf-8-sig")
imp.head(15).round(4)

In [ ]:
display(Image(filename=str(FIGURES / "17_shap_bar_regresion_pli.png")))
display(Image(filename=str(FIGURES / "16_shap_summary_regresion_pli.png")))

## 4. Control independiente: importancia por permutación

In [ ]:
pi = pd.read_csv(TABLES / "importancia_permutacion_regresion_pli.csv", encoding="utf-8-sig")
top10_shap = set(imp["variable"].head(10))
top10_perm = set(pi["variable"].head(10))
print("Coincidencia top-10:", len(top10_shap & top10_perm), "/ 10")
pi.head(15).round(4)

**Lectura.** Las cinco variables que más empujan la predicción del PLI son
**Fe, Mo, Co, Tl y Sb**; les siguen `ratio_Ca_Mg`, Al, la coordenada Norte, B y
K. La concordancia con la permutación es **9 de 10**, así que la explicación no
depende de la técnica.

La interpretación geoquímica encaja con un sistema mineralizado tipo
pórfido/epitermal: Fe como fase portadora y Mo-Co-Tl-Sb como trazadores de
alteración hidrotermal. Ninguno de los 8 metales del índice entra al modelo, de
modo que la señal viene del contexto litogénico, no de una fuga.

## 5. Contribución por muestra y muestras críticas

In [ ]:
contrib = pd.read_csv(PROCESSED / "shap_contribucion_por_muestra.csv", encoding="utf-8-sig")
contrib.head()

In [ ]:
import json
resumen_shap = json.loads((TABLES / "resumen_shap_regresion_pli.json").read_text(encoding="utf-8"))
pd.DataFrame(resumen_shap["muestras_criticas"])

## 6. Predicciones y ranking

In [ ]:
pred = pd.read_csv(PROCESSED / "predicciones_regresion_pli.csv", encoding="utf-8-sig")
pred[["PUNTO_NORMALIZADO", "DISTRITO", "PLI", "PLI_predicho"]].head(10)

In [ ]:
ranking = pd.read_csv(TABLES / "ranking_puntos_criticos.csv", encoding="utf-8-sig")
ranking

## 7. Zonas críticas

El plan pide localizar zonas, no solo puntos. Se agregan las muestras en
celdas UTM de 5 × 5 km y se ordenan por severidad normativa, límite inferior
del IC 95 % de Wilson del porcentaje en *Alto/Muy alto*, número de puntos y
PLI mediano.

In [ ]:
zonas = pd.read_csv(TABLES / "zonas_criticas.csv", encoding="utf-8-sig")
zonas[["GRID_5KM", "distrito", "n_muestras", "n_puntos", "nivel_riesgo",
       "pct_alto", "pct_alto_ic95_inf", "PLI_mediano",
       "metal_dominante_normativo"]]

**Cierre.** El ranking y las zonas críticas combinan severidad normativa, PLI
mediano y la variable SHAP dominante, de modo que cada foco llega con una
hipótesis de causa verificable en campo.